# URAE Tutorial: Non-Linear Finite Element Analysis (FEA)

This tutorial showcases both **exact closed-form symbolic solutions** and **numerical iterative Newton-Raphson methods** for non-linear solid mechanics and elasticity:

$$\mathbf{K}_{\text{tan}}(\mathbf{u}) \Delta \mathbf{u} = \mathbf{F}_{\text{ext}} - \mathbf{F}_{\text{int}}(\mathbf{u})$$

- **Part 1**: Closed-form analytical non-linear hyperelastic bar
- **Part 2**: Numerical Newton-Raphson 3D FEA with tangent stiffness matrix assembly and stress contour plotting

In [1]:
import urae

# =========================================================================
# PART 1: EXACT CLOSED-FORM SYMBOLIC NON-LINEAR ELASTICITY
# =========================================================================
x = urae.var("x")
E, A, L, F, alpha = 210000.0, 100.0, 1000.0, 15000.0, 0.002

# Non-linear axial displacement field u(x)
u_sym = (F / (A * E)) * x * (1.0 - 0.5 * alpha * (F / (A * E)) * x)
eps_sym = u_sym.diff(x)
sigma_sym = E * (eps_sym + alpha * (eps_sym**2))

print("Symbolic Displacement u(x) =", u_sym)
print("Symbolic Strain eps(x) =", eps_sym)
print("Symbolic Stress sigma(x) =", sigma_sym)
print("Tip Deflection u(L) =", float(u_sym.subs({x: L})), "mm")

In [2]:
# =========================================================================
# PART 2: NUMERICAL NON-LINEAR FEA (NEWTON-RAPHSON)
# =========================================================================
mesh = urae.fea.generate_beam_mesh(length=200.0, height=25.0, depth=15.0, elements=(12, 3, 3))
print(f"Generated Mesh: {mesh.num_nodes} nodes, {mesh.num_elements} hexahedral elements")

# Set material model and boundary constraints
material = urae.fea.HyperelasticMaterial(E=210000.0, nu=0.30, yield_stress=355.0)
solver = urae.fea.NonlinearNewtonRaphsonSolver(mesh=mesh, material=material)
solver.add_dirichlet_bc(face="x_min", dofs=[0, 1, 2], value=0.0)
solver.add_point_load(node_idx=mesh.num_nodes - 1, force=[0.0, -F, 0.0])

# Solve iterative tangent system
solution = solver.solve(max_iter=10, tol=1e-6)
print(f"Convergence achieved in {solution.iterations} iterations!")
print(f"Max von Mises Stress: {solution.max_von_mises:.2f} MPa")
print(f"Peak Tip Deflection: {solution.max_displacement:.4f} mm")

In [3]:
# Part 3: Visualize Stress Contours & Mesh Deformation
urae.visualizer.plot_fea_mesh(solution, color_field="von_mises", deformation_scale=5.0)